<a href="https://colab.research.google.com/github/ritashreemukherjee123/Project-1-Applied-Search-Intelligence-Google-Search-Ranking-Discoverability.-/blob/main/work/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

**Search Intelligence data Contract - An active, provable commitment written before any ML modeling.**

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

Best Data for Lane 2 are the warehouse release (`dim_content` + `fact_content_daily_performance`) or the starter dataset.
Here, we are using the warehouse dataset.

**"Unit of analysis"**: Formal Version of **"Grain"**.

Here, `dim_content` has **"one row per pseudonymized content item"** which is the Grain of this Table. `dim_content` deduplicates content into one row per pseudonymized content item.

In simple words, **One row = one piece of content (article, feed, comparison, etc.)** for a specific client and keyword.

`fact_content_daily_performance` has daily x client x content as Grain which means **one row per report date, pseudonymized client, and pseudonymized content item**. The data **cuts off the freshest 3 days**: report_date <= DATE_SUB(DATE('2026-07-03'), INTERVAL 3 DAY) (so daily facts run through 2026-06-30).


In [1]:
%pip -q install duckdb huggingface_hub

In [2]:
import os, getpass

# Token order: env var -> Colab Secret -> prompt (last resort).
# Use a Colab Secret named HF_TOKEN (the key panel on the left) so the prompt never
# fires: if Colab reconnects while a getpass prompt is open, the kernel waits on it
# forever ('Resuming execution...') and you have to restart the runtime.
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Ritashree')

Ritashree··········


**Important**: That count over the daily fact touched **Parquet metadata, not data** — it finished in seconds
even though the table has ~79M rows. That is the whole workflow: push the heavy lifting into
DuckDB SQL, bring only small results into pandas.

The `REL` variable defines the base URL for the dataset warehouse. The `TABLES` dictionary maps user-friendly table names to DuckDB SQL expressions that specify how to read various Parquet files from that Hugging Face repository, some of which use a `glob pattern(**/*.parquet)` to read multiple files within a directory.

In [3]:
##Connecting DuckDB to the release which would autheticate every query written
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')



dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


In [4]:
# Columns of dim_content

result_dim_content = con.sql(f"""
SELECT *
FROM {TABLES['dim_content']}
LIMIT 5;
""").fetchdf()
print(result_dim_content)

            client_hash_id           content_hash_id  \
0  client_04660893ae39614a  content_004de9653278b5a4   
1  client_04660893ae39614a  content_00dc5efae381b2ab   
2  client_04660893ae39614a  content_01410f2556c327ac   
3  client_04660893ae39614a  content_019f27f634053ca7   
4  client_04660893ae39614a  content_01efa71faea45dcc   

            keyword_hash_id           url_hash_id  keyword_char_count  \
0  keyword_e754999ab88dd9f2  url_d6091f18cf628794                  22   
1  keyword_4329d7aede8e208b  url_3a66d2f2e36823ca                  31   
2  keyword_9b08047d3d2a0406  url_809eda7a7e20b3b2                  22   
3  keyword_e7cec7ab1804c1c2  url_5fb42bafc4399861                  14   
4  keyword_56b0062a1d8b7524  url_ece0abc3e5fb75f9                  24   

   keyword_token_count  url_char_count content_created_date  \
0                    4             108           2026-05-30   
1                    6              95           2026-06-12   
2                    5             

This code performs a SQL query that joins two tables: `fact_query_90d`(aliased as t1) and `dim_content`(aliased as t2). It selects various columns from both tables, including client_hash_id, content_hash_id (from both tables, aliased to distinguish them), query_hash_id, content_type, content_created_date, content_updated_date, and query_char_count. The tables are joined on content_hash_id, which means it combines rows where the content identifier is the same in both.

In [7]:
duplicate_check = con.sql(f"""
SELECT report_date, client_hash_id, content_hash_id,
COUNT(*) AS n_rows FROM {TABLES['fact_daily_sample']} GROUP
BY report_date, client_hash_id, content_hash_id
HAVING COUNT(*) > 1 ORDER BY n_rows DESC
LIMIT 5;""").fetchdf()

print(duplicate_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  report_date           client_hash_id           content_hash_id  n_rows
0  2026-06-13  client_1a730cb2640a1abf  content_0a21add649629840       2
1  2026-06-13  client_1a730cb2640a1abf  content_0ffd552a735a66ab       2
2  2026-06-13  client_1a730cb2640a1abf  content_f85822ce54f90378       2
3  2026-06-13  client_1a730cb2640a1abf  content_ff51845ed161bfb4       2
4  2026-06-13  client_1a730cb2640a1abf  content_8f5be8e45452dda7       2


**Grain of fact_query_90d**: It seems to represent individual queries (query_hash_id) made by a client_hash_id in relation to a content_hash_id. This means each row likely corresponds to a specific query event.

In [5]:
#Join content facts to dim_content on content_hash_id.
result = con.sql(f"""
SELECT
    t1.client_hash_id,
    t1.content_hash_id AS fact_content_hash_id,
    t2.content_hash_id AS dim_content_hash_id,
    t1.query_hash_id,
    t1.query_char_count,
    t2.content_type,
    t2.content_created_date,
    t2.content_updated_date,
    t1.query_char_count

FROM
    {TABLES['fact_query_90d']} AS t1
JOIN
    {TABLES['dim_content']} AS t2
ON
    t1.content_hash_id = t2.content_hash_id
LIMIT 5;
""").fetchdf()
print(result)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

            client_hash_id      fact_content_hash_id  \
0  client_08a6a72ff48e62c0  content_5f6556f5a53e47f0   
1  client_08a6a72ff48e62c0  content_5f6c0644690c7bda   
2  client_08a6a72ff48e62c0  content_5f6c0644690c7bda   
3  client_08a6a72ff48e62c0  content_5f6c0644690c7bda   
4  client_08a6a72ff48e62c0  content_5f6c0644690c7bda   

        dim_content_hash_id           query_hash_id  query_char_count  \
0  content_5f6556f5a53e47f0  query_4225de77a704f13b                22   
1  content_5f6c0644690c7bda  query_5334a84c25d08ae8                28   
2  content_5f6c0644690c7bda  query_58d07c42a27e46a6                20   
3  content_5f6c0644690c7bda  query_871044f200f69372                16   
4  content_5f6c0644690c7bda  query_ba5ee754917c395e                13   

      content_type content_created_date content_updated_date  \
0  keyword article           2026-04-02           2026-05-20   
1  keyword article           2025-07-31           2026-02-25   
2  keyword article           202

In [8]:
# Join client-level facts to dim_clients on client_hash_id.
result_client_join = con.sql(f"""
SELECT
    t1.client_hash_id AS fact_client_hash_id,
    t1.content_hash_id,
    t1.query_hash_id,
    t2.client_hash_id AS dim_client_hash_id,
    t2.client_created_date,
    t1.query_char_count
FROM
    {TABLES['fact_query_90d']} AS t1
JOIN
    {TABLES['dim_clients']} AS t2
ON
    t1.client_hash_id = t2.client_hash_id
LIMIT 5;
""").fetchdf()
print(result_client_join)

       fact_client_hash_id           content_hash_id           query_hash_id  \
0  client_08a6a72ff48e62c0  content_447894f2faf0d2bc  query_58b1b001f839d699   
1  client_08a6a72ff48e62c0  content_447894f2faf0d2bc  query_922b8eca2a24cd34   
2  client_08a6a72ff48e62c0  content_447894f2faf0d2bc  query_9f0c36a6ae2a6a99   
3  client_08a6a72ff48e62c0  content_447894f2faf0d2bc  query_a032820b5467e996   
4  client_08a6a72ff48e62c0  content_447894f2faf0d2bc  query_ba1a2f131961c5da   

        dim_client_hash_id client_created_date  query_char_count  
0  client_08a6a72ff48e62c0          2025-05-26                17  
1  client_08a6a72ff48e62c0          2025-05-26                34  
2  client_08a6a72ff48e62c0          2025-05-26                16  
3  client_08a6a72ff48e62c0          2025-05-26                24  
4  client_08a6a72ff48e62c0          2025-05-26                18  


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*



In [ ]:
content_analysis_result = con.sql(f"""
SELECT

    t1.content_hash_id AS fact_content_hash_id,
    t2.content_hash_id AS dim_content_hash_id,
    t2.content_type,
    t2.content_created_date,
    t2.content_updated_date
FROM
    {TABLES['fact_daily_sample']} AS t1
JOIN
    {TABLES['dim_content']} AS t2
ON
    t1.content_hash_id = t2.content_hash_id
LIMIT 5;
""").fetchdf()
print(content_analysis_result)

       fact_content_hash_id       dim_content_hash_id     content_type  \
0  content_1a6296faee432dae  content_1a6296faee432dae  keyword article   
1  content_73f21e612565035a  content_73f21e612565035a  keyword article   
2  content_5a5be514ff559598  content_5a5be514ff559598  keyword article   
3  content_05b377d0c8a5cfd8  content_05b377d0c8a5cfd8  keyword article   
4  content_dc34c661d63e55a9  content_dc34c661d63e55a9  keyword article   

  content_created_date content_updated_date  
0           2025-02-05           2026-05-20  
1           2025-02-05           2026-05-20  
2           2025-02-05           2026-05-20  
3           2025-02-05           2026-05-20  
4           2025-02-05           2026-05-20  


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

**Grain**: Already stated which are the grain or unit of analysis are for each tables of data.



In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

#Grain of dim_content
result_dim_content = con.sql(f"""
SELECT content_hash_id, content_type
FROM {TABLES['dim_content']}
LIMIT 5;
""").fetchdf()
print(result_dim_content)

            content_hash_id     content_type
0  content_004de9653278b5a4  keyword article
1  content_00dc5efae381b2ab  keyword article
2  content_01410f2556c327ac  keyword article
3  content_019f27f634053ca7  keyword article
4  content_01efa71faea45dcc  keyword article


In [ ]:
#Grain of fact_daily_sample, with time window (report date, month of report)
# showing 'daily x client x content' grain.

result_fact_daily_sample = con.sql(f"""
SELECT report_date, client_hash_id, content_hash_id
FROM {TABLES['fact_daily_sample']}
LIMIT 5;
""").fetchdf()
print(result_fact_daily_sample)

  report_date           client_hash_id           content_hash_id
0  2026-06-01  client_3ffa76342f366962  content_1a6296faee432dae
1  2026-06-01  client_3ffa76342f366962  content_73f21e612565035a
2  2026-06-01  client_3ffa76342f366962  content_5a5be514ff559598
3  2026-06-01  client_3ffa76342f366962  content_05b377d0c8a5cfd8
4  2026-06-01  client_3ffa76342f366962  content_dc34c661d63e55a9


**Counts**, here, specify slice’s row count and date span, and availability.

In [ ]:
unique_content_count = con.sql(f"""
SELECT COUNT(DISTINCT content_hash_id)
FROM {TABLES['dim_content']}
""").fetchone()[0]
print(f"Unique content_hash_id in dim_content: {unique_content_count:,} (should match total dim_content rows: {con.sql(f'SELECT COUNT(*) FROM {TABLES['dim_content']}').fetchone()[0]:,})")

# Verify the count for fact_daily_sample
fact_daily_sample_row_count = con.sql(f"""
SELECT COUNT(*)
FROM {TABLES['fact_daily_sample']}
""").fetchone()[0]
print(f"Total rows in fact_daily_sample: {fact_daily_sample_row_count:,}")

# Verify the grain of fact_daily_sample by counting unique combinations of (report_date, client_hash_id, content_hash_id)
unique_fact_daily_sample_grain_count = con.sql(f"""
SELECT COUNT(DISTINCT (report_date, client_hash_id, content_hash_id))
FROM {TABLES['fact_daily_sample']}
""").fetchone()[0]
print(f"Unique (report_date, client_hash_id, content_hash_id) combinations in fact_daily_sample: {unique_fact_daily_sample_grain_count:,}")

# If the unique grain count matches the total row count, it confirms the grain
if unique_fact_daily_sample_grain_count == fact_daily_sample_row_count:
    print("The grain of 'daily x client x content' for fact_daily_sample is confirmed by unique combinations.")
else:
    print("The grain of 'daily x client x content' for fact_daily_sample does not uniquely identify all rows.")

Unique content_hash_id in dim_content: 519,606 (should match total dim_content rows: 519,606)
Total rows in fact_daily_sample: 11,694,072


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Unique (report_date, client_hash_id, content_hash_id) combinations in fact_daily_sample: 11,687,682
The grain of 'daily x client x content' for fact_daily_sample does not uniquely identify all rows.


### Missing Values Check

Checking for `NULL` values in critical identifier columns (`content_hash_id`, `client_hash_id`, `report_date`) for both `dim_content` and `fact_daily_sample` to ensure data availability.

In [9]:
# Check for missing content_hash_id in dim_content
missing_dim_content_id = con.sql(f"""
SELECT COUNT(*)
FROM {TABLES['dim_content']}
WHERE content_hash_id IS NULL
""").fetchone()[0]
print(f"Missing content_hash_id in dim_content: {missing_dim_content_id:,}")

# Check for missing identifiers in fact_daily_sample
missing_fact_daily_sample_ids = con.sql(f"""
SELECT
    SUM(CASE WHEN report_date IS NULL THEN 1 ELSE 0 END) AS missing_report_date,
    SUM(CASE WHEN client_hash_id IS NULL THEN 1 ELSE 0 END) AS missing_client_hash_id,
    SUM(CASE WHEN content_hash_id IS NULL THEN 1 ELSE 0 END) AS missing_content_hash_id
FROM {TABLES['fact_daily_sample']}
""").fetchdf()
print("\nMissing identifiers in fact_daily_sample:")
print(missing_fact_daily_sample_ids)

Missing content_hash_id in dim_content: 0

Missing identifiers in fact_daily_sample:
   missing_report_date  missing_client_hash_id  missing_content_hash_id
0                  0.0                     0.0                      0.0


### Date Windows (Time Windows)


In [10]:
# Get the min and max report_date from fact_daily_sample
date_window_fact_daily_sample = con.sql(f"""
SELECT MIN(report_date), MAX(report_date)
FROM {TABLES['fact_daily_sample']}
""").fetchdf()
print(f"Report date window for fact_daily_sample: {date_window_fact_daily_sample.iloc[0, 0]} to {date_window_fact_daily_sample.iloc[0, 1]}")



Report date window for fact_daily_sample: 2026-06-01 00:00:00 to 2026-06-30 00:00:00


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

1. **Unbalanced History**:
   - The `fact_daily` table explicitly 'cuts off the freshest 3 days' (e.g., data only up to 2026-06-30).

2. **GSC-only early rows (Google Search Console-only early rows)**:
    - While not explicitly stated as Google Search Console (GSC) data, the context of 'Search Intelligence'. But, even if GSC is the primary source, it would depend on the GSC's data retention policies.

3. **Window Overlaps**: No, the data does not state any window overlap as distinct dates are seen.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.



# Get report_date from fact_daily_sample to show its range
print("Fact Daily Sample Report Dates:")
unbalanced_history_fact_daily = con.sql(f"""
SELECT MIN(report_date) AS min_report_date, MAX(report_date) AS max_report_date
FROM {TABLES['fact_daily_sample']}
""").fetchdf()
print(unbalanced_history_fact_daily)

# Get content_updated_date from dim_content to show its range
print("\nDim Content Updated Dates:")
unbalanced_history_dim_content = con.sql(f"""
SELECT MIN(content_updated_date) AS min_content_updated_date, MAX(content_updated_date) AS max_content_updated_date
FROM {TABLES['dim_content']}
""").fetchdf()
print(unbalanced_history_dim_content)

# This comparison helps illustrate 'unbalanced history' where fact data might have a cutoff

Fact Daily Sample Report Dates:
  min_report_date max_report_date
0      2026-06-01      2026-06-30

Dim Content Updated Dates:
  min_content_updated_date max_content_updated_date
0               2024-10-28               2026-07-06


In [ ]:
# Window Overlaps

# 1. Get the min and max report_date from fact_daily_sample
min_max_daily_sample = con.sql(f"""
SELECT MIN(report_date) AS min_date, MAX(report_date) AS max_date
FROM {TABLES['fact_daily_sample']}
""").fetchdf()
print(f"Fact Daily Sample Date Range: {min_max_daily_sample['min_date'].iloc[0]} to {min_max_daily_sample['max_date'].iloc[0]}")


Fact Daily Sample Date Range: 2026-06-01 00:00:00 to 2026-06-30 00:00:00


###Target/Proxy Field


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.